In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
df = pd.read_csv('dataset_mood_smartphone.csv', index_col = 0)
df.head()

df['time'] = pd.to_datetime(df['time'])

print(df.shape[0])

376912


In [16]:
# Duration-type variable labels
duration_labels = [
    "screen",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
    "appCat.social", "appCat.travel", "appCat.unknown",
    "appCat.utilities", "appCat.weather"
]

# Detect negative durations
neg_durations = df[
    (df["variable"].isin(duration_labels)) &
    (df["value"] < 0)
]

print(f"Negative duration rows found: {neg_durations.shape[0]}")
print(neg_durations[["id", "variable", "value"]].head())

# Remove them
df_no_neg = df.drop(index=neg_durations.index)

print(df_no_neg.shape[0])


Negative duration rows found: 4
             id              variable      value
151512  AS14.02        appCat.builtin    -44.689
622771  AS14.07        appCat.builtin -82798.871
754571  AS14.12        appCat.builtin     -1.218
484694  AS14.02  appCat.entertainment     -0.011
376908


In [17]:
# Identify exact duplicates
duplicates_mask = df_no_neg.duplicated(keep=False)
duplicate_rows = df_no_neg[duplicates_mask].sort_values(by=["id", "time", "variable"])

print(f"Exact duplicate rows: {duplicate_rows.shape[0]}")
print(duplicate_rows.head(10))

# Remove duplicates (keep first)
df_no_neg_dup = df_no_neg.drop_duplicates(keep="first")

print(f"Rows after removing duplicates: {df_no_neg_dup.shape[0]}")


Exact duplicate rows: 12
            id                time            variable  value
11472  AS14.01 2014-04-27 20:00:00  circumplex.valence    1.0
11473  AS14.01 2014-04-27 20:00:00  circumplex.valence    1.0
11869  AS14.03 2014-05-03 11:00:00  circumplex.valence    1.0
11870  AS14.03 2014-05-03 11:00:00  circumplex.valence    1.0
7392   AS14.12 2014-03-30 11:00:00  circumplex.arousal   -1.0
7393   AS14.12 2014-03-30 11:00:00  circumplex.arousal   -1.0
15043  AS14.24 2014-05-11 11:00:00  circumplex.valence    1.0
15044  AS14.24 2014-05-11 11:00:00  circumplex.valence    1.0
15253  AS14.25 2014-04-27 11:00:00  circumplex.valence    0.0
15254  AS14.25 2014-04-27 11:00:00  circumplex.valence    0.0
Rows after removing duplicates: 376902


In [18]:

# Get unique user IDs
user_ids = df_no_neg_dup["id"].unique()
print(f"Found {len(user_ids)} unique users.")

# Collect all real violations
all_violations = []

for user in user_ids:
    print(f"Processing user: {user}")
    
    screen_df = df_no_neg_dup[
        (df_no_neg_dup["variable"] == "screen") &
        (df_no_neg_dup["id"] == user)
    ]
    
    app_df = df_no_neg_dup[
        (df_no_neg_dup["variable"].isin(duration_labels)) &
        (df_no_neg_dup["variable"] != "screen") &
        (df_no_neg_dup["id"] == user)
    ]

    invalid_app_rows = []

    for idx, row in screen_df.iterrows():
        start_time = row["time"]
        duration = row["value"]
        end_time = start_time + pd.to_timedelta(duration, unit="s")

        apps_in_window = app_df[
            (app_df["time"] >= start_time) &
            (app_df["time"] <= end_time)
        ].copy()

        apps_in_window["screen_start"] = start_time
        apps_in_window["screen_end"] = end_time
        apps_in_window["screen_duration"] = duration

        over_limit = apps_in_window[apps_in_window["value"] > duration]

        if not over_limit.empty:
            invalid_app_rows.append(over_limit)

    if invalid_app_rows:
        result_df = pd.concat(invalid_app_rows)
        result_df["overuse"] = result_df["value"] - result_df["screen_duration"]
        real_violations = result_df[result_df["overuse"] > 5]
        all_violations.append(real_violations)

# Combine all violations across users
final_violations = pd.concat(all_violations)

# Remove them from df_cleaned
before = df_no_neg_dup.shape[0]
df_cleaned = df_no_neg_dup.drop(index=final_violations.index)
after = df_cleaned.shape[0]

# Report
print(f"Removed {before - after} rows with impossible app durations")
print(df_cleaned.shape[0])


Found 27 unique users.
Processing user: AS14.01
Processing user: AS14.02
Processing user: AS14.03
Processing user: AS14.05
Processing user: AS14.06
Processing user: AS14.07
Processing user: AS14.08
Processing user: AS14.09
Processing user: AS14.12
Processing user: AS14.13
Processing user: AS14.14
Processing user: AS14.15
Processing user: AS14.16
Processing user: AS14.17
Processing user: AS14.19
Processing user: AS14.20
Processing user: AS14.23
Processing user: AS14.24
Processing user: AS14.25
Processing user: AS14.26
Processing user: AS14.27
Processing user: AS14.28
Processing user: AS14.29
Processing user: AS14.30
Processing user: AS14.31
Processing user: AS14.32
Processing user: AS14.33
Removed 891 rows with impossible app durations
376011


In [19]:
print(f"Total violations found (any overuse > 0): {sum([len(df) for df in all_violations])}")
print(df_cleaned.shape[0])


Total violations found (any overuse > 0): 891
376011


In [20]:
# Mood
mood_outliers = df_cleaned[
    (df_cleaned["variable"] == "mood") &
    ((df_cleaned["value"] < 1) | (df_cleaned["value"] > 10))
]

# Activity
activity_outliers = df_cleaned[
    (df_cleaned["variable"] == "activity") &
    ((df_cleaned["value"] < 0) | (df_cleaned["value"] > 1))
]

# Arousal / valence
arousal_outliers = df_cleaned[
    (df_cleaned["variable"] == "circumplex.arousal") &
    ((df_cleaned["value"] < -2) | (df_cleaned["value"] > 2))
]

valence_outliers = df_cleaned[
    (df_cleaned["variable"] == "circumplex.valence") &
    ((df_cleaned["value"] < -2) | (df_cleaned["value"] > 2))
]

# Call / SMS
binary_outliers = df_cleaned[
    (df_cleaned["variable"].isin(["call", "sms"])) &
    (~df_cleaned["value"].isin([0, 1]))
]

print(f"Mood outliers: {mood_outliers.shape[0]}")
print(f"Activity outliers: {activity_outliers.shape[0]}")
print(f"Arousal outliers: {arousal_outliers.shape[0]}")
print(f"Valence outliers: {valence_outliers.shape[0]}")
print(f"Call/SMS outliers: {binary_outliers.shape[0]}")

Mood outliers: 0
Activity outliers: 0
Arousal outliers: 0
Valence outliers: 0
Call/SMS outliers: 0


# Imoutation method

In [21]:
# df_cleaned['value'] = df_cleaned.groupby(['id', 'variable'])['value'].fillna(method='ffill')
# df_cleaned['value'] = df_cleaned.groupby(['id', 'variable'])['value'].fillna(method='bfill')

In [22]:
# missing = df_cleaned.isnull().sum()
# missing

In [23]:
#add new date column
df_cleaned['date'] = df_cleaned['time'].dt.date
daily_median = df_cleaned.groupby(['id', 'variable', 'date'])['value'].transform('median')
df_cleaned['value'] = df_cleaned['value'].fillna(daily_median)

In [24]:
missing_rows = df_cleaned[df_cleaned['value'].isnull()]
missing_rows

,id,time,variable,value,date
15039,AS14.24,2014-05-10 09:00:00,circumplex.valence,NaN,2014-05-10
15040,AS14.24,2014-05-10 12:00:00,circumplex.valence,NaN,2014-05-10
15041,AS14.24,2014-05-10 18:00:00,circumplex.valence,NaN,2014-05-10
15042,AS14.24,2014-05-10 21:00:00,circumplex.valence,NaN,2014-05-10
15168,AS14.24,2014-06-07 12:00:00,circumplex.valence,NaN,2014-06-07


In [25]:
daily_medians = df_cleaned.groupby(['id', 'variable', 'date'])['value'].median().reset_index(name='daily_median')

# shift median to previous day (per id & variable)
daily_medians['prev_day'] = daily_medians.groupby(['id', 'variable'])['daily_median'].shift(1).copy()
daily_medians['next_day'] = daily_medians.groupby(['id', 'variable'])['daily_median'].shift(-1).copy()

# merge shifted values back to the main df
df_cleaned = df_cleaned.merge(daily_medians[['id', 'variable', 'date', 'prev_day', 'next_day']], 
              on=['id', 'variable', 'date'], how='left')

# fill  NaN values in using the previous day's median
df_cleaned['value'] = df_cleaned['value'].fillna((df_cleaned['prev_day']+df_cleaned['next_day'])/2)

# drop the now redundand column
df_cleaned.drop(columns=['prev_day'], inplace=True)
df_cleaned.drop(columns=['next_day'], inplace=True)

In [26]:
missing = df_cleaned.isnull().sum()
missing

id          0
time        0
variable    0
value       0
date        0
dtype: int64

In [27]:
df_cleaned.to_csv('df_cleaned.csv', index=False)
